# Module 4 · Recurrent Networks, LSTMs & 1D CNNs for Text

**Track:** Natural Language Processing (0 to Masterclass)  
**Prerequisites:** NLP 02 (Embeddings), DL 01 (Neural Network Mathematics)  
**Difficulty:** ⭐⭐⭐⭐ Advanced  
**Estimated Time:** 180–240 minutes

---



## 🎓 Socratic Deep Check

> *"Vanilla RNNs are logically sound for processing sequences, yet they practically fail for any dependency longer than ~10-20 steps. If the math for backpropagation (BPTT) is consistent, why does the gradient 'die' instead of flowing? Is it a hardware limitation or a fundamental mathematical property of recurrence?"*

> *"Furthermore, if the problem with RNNs is their serial, step-by-step nature, could we instead treat a sentence like an image — sweeping a filter across words to detect local patterns? And what are the trade-offs of this parallelism?"*

---

## The Core Problem: Why Recurrence?

1. **Variable Length Input**: Traditional MLPs require fixed-size inputs. Natural language is dynamic — a sentence can be 3 words or 3,000.
2. **The Stationary Property**: The grammar rules that apply at the beginning of a sentence often apply at the end. We need **Shared Weights** to exploit this temporal invariance.
3. **Contextual Memory**: A word's meaning depends on its history. To understand "it" in "The cat ate the mouse and it died," you need to retain state from several steps back.

---

## 📑 Table of Contents

* [🎓 Socratic Deep Check](#socratic-deep-check)
* [The Core Problem: Why Recurrence?](#the-core-problem-why-recurrence)
  * [🎓 The Senior Perspective: Evolutionary Architecture](#the-senior-perspective-evolutionary-architecture)
* [1 · Vanilla RNN: The Recurrence Formula](#1-vanilla-rnn-the-recurrence-formula)
  * [Backpropagation Through Time (BPTT)](#backpropagation-through-time-bptt)
  * [🔬 Mathematical Proof: Vanishing Gradients](#mathematical-proof-vanishing-gradients)
* [1b · Scaling Recurrence: Truncated BPTT and Stateful RNNs](#1b-scaling-recurrence-truncated-bptt-and-stateful-rnns)
  * [The RAM Limitation: Infinite Memory vs. Finite Hardware](#the-ram-limitation-infinite-memory-vs-finite-hardware)
  * [Truncated BPTT (TBPTT)](#truncated-bptt-tbptt)
  * [Stateful Training: The `.detach()` Secret](#stateful-training-the-detach-secret)
* [2 · LSTM: Long Short-Term Memory](#2-lstm-long-short-term-memory)
  * [The Gating Mechanism](#the-gating-mechanism)
* [2b · GRU: Gated Recurrent Unit](#2b-gru-gated-recurrent-unit)
* [3 · Bidirectional RNNs (BiLSTMs)](#3-bidirectional-rnns-bilstms)
* [4 · Production Pipeline: POS Tagging with BiLSTM + Gradient Clipping](#4-production-pipeline-pos-tagging-with-bilstm-gradient-clipping)
  * [Why Pack Sequences?](#why-pack-sequences)
  * [⚠️ Gradient Clipping: The Non-Negotiable Safeguard](#gradient-clipping-the-non-negotiable-safeguard)
  * [4.2 · Tensor Memory Layout: Packed Padded Sequences](#42-tensor-memory-layout-packed-padded-sequences)
    * [1. The "Padding Noise" Problem](#1-the-padding-noise-problem)
    * [2. The Solution: Packing Sequences](#2-the-solution-packing-sequences)
    * [3. Engineering Pattern: Pack → Process → Unpack](#3-engineering-pattern-pack-process-unpack)
  * [4.4 · The Missing Link for Sequence Tagging: Conditional Random Fields (CRFs)](#44-the-missing-link-for-sequence-tagging-conditional-random-fields-crfs)
  * [The Structural Flaw in BiLSTM Taggers](#the-structural-flaw-in-bilstm-taggers)
  * [Enter the CRF Layer](#enter-the-crf-layer)
  * [Inference: Viterbi Decoding](#inference-viterbi-decoding)
* [5 · Unaligned Sequences: Connectionist Temporal Classification (CTC) Loss](#5-unaligned-sequences-connectionist-temporal-classification-ctc-loss)
  * [🎓 Socratic Deep Check](#socratic-deep-check)
  * [The Alignment Problem](#the-alignment-problem)
  * [Connectionist Temporal Classification (CTC)](#connectionist-temporal-classification-ctc)
    * [1. The Invention of the "Blank Token" (`-`)](#1-the-invention-of-the-blank-token)
    * [2. The Mechanism: Marginalizing over Paths](#2-the-mechanism-marginalizing-over-paths)
* [6 · 1D CNNs for Text (TextCNN): When Parallelism Beats Memory](#6-1d-cnns-for-text-textcnn-when-parallelism-beats-memory)
  * [🔬 TextCNN: Worked Example — What Does Each Kernel Detect?](#textcnn-worked-example-what-does-each-kernel-detect)
* [7 · Performance & The Evolution to Attention](#7-performance-the-evolution-to-attention)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: The Forget Gate](#q1-the-forget-gate)
  * [Q2: Bidirectionality](#q2-bidirectionality)
  * [Q3: Gradient Clipping](#q3-gradient-clipping)
  * [Q4: TextCNN vs LSTM](#q4-textcnn-vs-lstm)
  * [Q5: GRU vs LSTM](#q5-gru-vs-lstm)


### 🎓 The Senior Perspective: Evolutionary Architecture

A Senior Engineer views RNN architectures as a bridge. They understand that while Transformers dominate large-scale NLP, RNNs introduced the fundamental concept of **distributed state** and **gating mechanisms** (LSTMs/GRUs) which paved the way for the internal logic of Attention blocks. Additionally, 1D CNNs for text showed that not every sequence problem *requires* sequential computation — a lesson that foreshadowed multi-head attention's parallel nature.

**Mental Model:**
- **Vanilla RNN**: Reading word-by-word with a single, finite memory bucket. If the bucket overflows or leaks (vanishing gradient), you forget the beginning.
- **LSTM**: Reading with a notebook (Cell State) and an eraser (Forget Gate). You decide exactly what to keep and what to trash.
- **GRU**: A lean version of LSTM — fewer gates, same spirit, often faster.
- **1D CNN (TextCNN)**: Reading with a sliding magnifying glass. Very fast and parallelizable, but only sees local windows — like an n-gram detector on steroids.
- **Transformer**: Having the whole book open on a table and "looking" at relevant words for every word produced (Search-and-Select model).

---

In [ ]:
!pip install -q torch numpy matplotlib tqdm

## 1 · Vanilla RNN: The Recurrence Formula

In a standard Feed-Forward network, each input $x$ is processed independently. In an RNN, we maintain a hidden state $h_t$ that carries information from the past ($h_{t-1}$).

$$
\begin{aligned}
h_t &= \phi(W_{hh} h_{t-1} + W_{xh} x_t + b_1) \\
y_t &= W_{hy} h_t + b_2
\end{aligned}
$$

Where $\phi$ is typically `tanh` or `ReLU`.

### Backpropagation Through Time (BPTT)

To compute the gradient of the loss $L$ with respect to the recurrent weights $W_{hh}$, we must use the chain rule across all previous time steps:

$$\frac{\partial L_t}{\partial W_{hh}} = \sum_{k=1}^t \frac{\partial L_t}{\partial h_t} \cdot \frac{\partial h_t}{\partial h_k} \cdot \frac{\partial h_k}{\partial W_{hh}}$$

The "middle term" $\frac{\partial h_t}{\partial h_k}$ is the culprit for the vanishing gradient problem.

### 🔬 Mathematical Proof: Vanishing Gradients

Let's look at the term $\frac{\partial h_t}{\partial h_k}$ for some $k < t$. By the chain rule:

$$\frac{\partial h_t}{\partial h_k} = \prod_{j=k+1}^t \frac{\partial h_j}{\partial h_{j-1}}$$

Each individual Jacobian $\frac{\partial h_j}{\partial h_{j-1}}$ is:
$$\frac{\partial h_j}{\partial h_{j-1}} = W_{hh}^T \cdot \text{diag}(\phi'(z_j))$$

Where $z_j = W_{hh} h_{j-1} + W_{xh} x_j + b_1$. If we take the norm:

$$\| \frac{\partial h_t}{\partial h_k} \| \le \prod_{j=k+1}^t \| W_{hh}^T \| \cdot \| \text{diag}(\phi'(z_j)) \|$$

**Crucial Insight:**
1. For the `tanh` activation, the derivative $\phi'(z) \in (0, 1]$.
2. If the largest eigenvalue (spectral radius) of $W_{hh}$ is less than 1, the product $\prod \| W_{hh} \|$ will shrink exponentially as $t-k$ increases.
3. Result: $\frac{\partial L_t}{\partial h_k} \to 0$ for large $t - k$. The model **cannot learn dependencies** that span beyond a few steps.

🎓 *Socratic Probe: Notice that the same mathematics also implies the opposite failure. What happens if the spectral radius of $W_{hh}$ is **greater than 1**? And what engineering technique directly addresses that symmetric catastrophe?*

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

def plot_gradient_flow(seq_len=50, w_scale=0.8):
    """Simulate the gradient flow backward through a vanilla RNN."""
    # Simplified: d_h(t)/d_h(t-1) = W * phi'(z)
    # We'll assume phi'(z) is constant for visualization (tanh max derivative is 1)
    
    # Random weight matrix with specific scale
    W = np.random.randn(64, 64) * w_scale / np.sqrt(64)
    
    norms = []
    grad = np.random.randn(64, 1) # Initial grad at step T
    
    for _ in range(seq_len):
        grad = W.T @ grad
        norms.append(np.linalg.norm(grad))
    
    return norms

plt.figure(figsize=(10, 5))
plt.plot(plot_gradient_flow(w_scale=0.5), label='||W|| = 0.5 (Vanishing)', color='#3498db', lw=2)
plt.plot(plot_gradient_flow(w_scale=1.1), label='||W|| = 1.1 (Exploding)', color='#e74c3c', lw=2)
plt.yscale('log')
plt.xlabel('Steps Backward in Time')
plt.ylabel('Gradient Norm')
plt.title('The Vanishing/Exploding Gradient Physics')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 1b · Scaling Recurrence: Truncated BPTT and Stateful RNNs

### The RAM Limitation: Infinite Memory vs. Finite Hardware
In standard **Backpropagation Through Time (BPTT)**, we treat the entire sequence of length $T$ as one giant computational graph. To calculate the gradient, the autograd engine must store every intermediate hidden state $h_t$ for $t=1 \dots T$ in memory (RAM/VRAM) to perform the chain rule.

**The Bottleneck:**
- If you are processing a 50-page book $(\sim 25,000\text{ tokens})$, the computational graph becomes so massive that it exceeds any modern GPU/TPU memory. 
- $O(T)$ space complexity makes naive RNN training on very long documents impossible.

### Truncated BPTT (TBPTT)
To solve this, we use **Truncated BPTT**. We split the sequence into smaller **chunks** (e.g., $k=100$ steps). 
1. **Forward Pass**: Process $k$ steps.
2. **Backpropagation**: Calculate loss and update weights *only* for these $k$ steps.
3. **State Handoff**: Pass the final hidden state $h_k$ as the *initial* state for the next chunk of $k$ steps.

### Stateful Training: The `.detach()` Secret
To make TBPTT work, the model must be **Stateful**. We want the model to "remember" the previous chunk's context, but we **cannot** let the computational graph span back into that previous chunk (otherwise we're back to $O(T)$ memory). 

In PyTorch, we solve this by calling `.detach()` on the hidden states. This keeps the vector values but cuts the history of how they were calculated, preventing the gradient from flowing back into the previous chunk.


In [ ]:
# --- Elite Production Pattern: Stateful RNN Training Loop ---

def train_stateful(model, data, chunk_len=100):
    model.train()
    hidden = None  # Initial state (None defaults to zeros)
    
    # Iterate through chunks of a long document
    for i in range(0, data.size(0) - chunk_len, chunk_len):
        # 1. Slice the document into a tractable chunk
        inputs = data[i : i + chunk_len]
        targets = data[i+1 : i + chunk_len + 1]
        
        # 2. Forward pass through the chunk
        # If hidden is None, LSTM/RNN creates a fresh zero-state.
        # If hidden is from the previous step, context flows forward!
        output, hidden = model(inputs, hidden)
        
        # 3. CRITICAL STEP: Truncate the history!
        # h.detach() and c.detach() stop the computational graph from 
        # growing infinitely. The model 'remembers' the values of the 
        # previous chunk but does NOT backpropagate through it.
        if isinstance(hidden, tuple): # For LSTM: (h, c)
            hidden = tuple(state.detach() for state in hidden)
        else: # For RNN/GRU: h
            hidden = hidden.detach()
            
        # 4. Standard optimization
        loss = criterion(output, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


## 2 · LSTM: Long Short-Term Memory

LSTMs (Hochreiter & Schmidhuber, 1997) solve vanishing gradients by introducing a **Cell State** ($C_t$) that acts as a protected conveyor belt. Gradients can flow through $C_t$ for many steps without being repeatedly multiplied by $W_{hh}$, thanks to the **Forget Gate**.

### The Gating Mechanism

1. **Forget Gate** ($f_t$): $\sigma(W_f[h_{t-1}, x_t] + b_f)$ — Decides what to drop from memory.
2. **Input Gate** ($i_t$): $\sigma(W_i[h_{t-1}, x_t] + b_i)$ — Decides what new info to store.
3. **Candidate State** ($\tilde{C}_t$): $\tanh(W_C[h_{t-1}, x_t] + b_C)$ — The actual new info.
4. **Output Gate** ($o_t$): $\sigma(W_o[h_{t-1}, x_t] + b_o)$ — Decides what to reveal as the hidden state.

**State Updates:**
$$
\begin{aligned}
C_t &= f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \\
h_t &= o_t \odot \tanh(C_t)
\end{aligned}
$$

The linear addition in $C_t$ update is the **"gradient highway"** — unlike $h_t$ which passes through repeated non-linearities, $C_t$ is updated additively. Gradients can flow backward through this addition with a magnitude of 1 wherever $f_t \approx 1$, breaking the exponential decay chain that kills Vanilla RNNs.

In [ ]:
class LSTMCell(nn.Module):
    """Elite Implementation: LSTM Cell from Scratch.
    
    Production Insight: We fuse all four gates (f, i, c_tilde, o) into a
    single Linear layer. This is a critical optimization — a single large
    matrix multiplication (4*H x D+H) is dramatically faster on a GPU than
    four separate smaller ones, due to CUDA kernel launch overhead and
    memory bandwidth efficiency.
    """
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # Combine all gates into one linear layer (f, i, C_tilde, o)
        self.xh = nn.Linear(input_dim + hidden_dim, 4 * hidden_dim)
        
    def forward(self, x: torch.Tensor, states: tuple) -> tuple:
        (h_prev, c_prev) = states
        combined = torch.cat([x, h_prev], dim=1)
        
        gates = self.xh(combined)
        f, i, c_tilde, o = gates.chunk(4, dim=1)
        
        f = torch.sigmoid(f)
        i = torch.sigmoid(i)
        c_tilde = torch.tanh(c_tilde)
        o = torch.sigmoid(o)
        
        # The gradient highway: additive update on C!
        c_curr = f * c_prev + i * c_tilde
        h_curr = o * torch.tanh(c_curr)
        
        return h_curr, c_curr

print("LSTM Cell instantiated with gate chunking strategy for GPU efficiency.")

## 2b · GRU: Gated Recurrent Unit

GRUs (Cho et al., 2014) are a popular alternative to LSTMs. They are computationally cheaper because they have fewer gates (2 instead of 3) and merge the cell state into the hidden state.

**Equations:**
1. **Update Gate** ($z_t$): $\sigma(W_z[h_{t-1}, x_t])$
2. **Reset Gate** ($r_t$): $\sigma(W_r[h_{t-1}, x_t])$
3. **Candidate State** ($\tilde{h}_t$): $\tanh(W_h[r_t \odot h_{t-1}, x_t])$
4. **Hidden State**: $h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$

| Feature | LSTM | GRU |
|---------|------|-----|
| Gates | 3 | 2 |
| Parameters | Higher | Lower (~3/4) |
| Speed | Slower | Faster |
| Performance | Often slightly better on long seq | Often similar/better on short seq |

In [ ]:
class GRUCell(nn.Module):
    """Elite Implementation: GRU Cell from Scratch.
    
    The Reset Gate controls long-term dependencies: when r_t ≈ 0, the
    candidate state ignores the previous hidden state entirely, allowing
    the model to 'reset' to a clean slate — equivalent to re-reading
    a new paragraph in a document.
    """
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.update_gate = nn.Linear(input_dim + hidden_dim, hidden_dim)
        self.reset_gate  = nn.Linear(input_dim + hidden_dim, hidden_dim)
        self.candidate   = nn.Linear(input_dim + hidden_dim, hidden_dim)
        
    def forward(self, x: torch.Tensor, h_prev: torch.Tensor) -> torch.Tensor:
        combined = torch.cat([x, h_prev], dim=1)
        
        z = torch.sigmoid(self.update_gate(combined))
        r = torch.sigmoid(self.reset_gate(combined))
        
        # Reset Gate modulates how much of the previous hidden state to use
        # in forming the candidate — the key insight of GRU.
        combined_reset = torch.cat([x, r * h_prev], dim=1)
        h_tilde = torch.tanh(self.candidate(combined_reset))
        
        h_curr = (1 - z) * h_prev + z * h_tilde
        return h_curr

print("GRU Cell implemented. Note the 'Reset Gate' which allows the model to drop state entirely.")

## 3 · Bidirectional RNNs (BiLSTMs)

In many NLP tasks, future context is just as important as the past. Consider:
> "I went to the **bank** to fish."
> "I went to the **bank** to deposit money."

A unidirectional LSTM reading from left-to-right only sees "I went to the..." when it reaches "bank". It needs context to the right of the word to disambiguate.

**Architecture:** Two independent LSTMs (Forward & Backward) whose outputs are concatenated.
$$h_t = [\vec{h}_t ; \overleftarrow{h}_t]$$

**Note:** BiRNNs cannot be used for causal tasks (Autoregressive Lexical Generation) because the future is not yet known.

## 4 · Production Pipeline: POS Tagging with BiLSTM + Gradient Clipping

Sequence tagging (assigning a label to *every* token) is the quintessential RNN task. We will implement it with **Packed Sequences** for optimal performance and **Gradient Clipping** for training stability.

### Why Pack Sequences?
In a batch, sequences have variable lengths. Padding (filling with zeros) wastes computation. `pack_padded_sequence` tells PyTorch to skip the zeros during the recurrent steps, significantly speeding up training.

---

### ⚠️ Gradient Clipping: The Non-Negotiable Safeguard

While LSTMs largely solve the **vanishing** gradient problem, they do not eliminate **exploding** gradients. During BPTT, a particularly "unlucky" mini-batch — where the gradient contributions from many time steps happen to add constructively — can produce a catastrophically large gradient update. This single step can:
- Shoot the weights to NaN, destroying the entire training run.
- Cause the optimizer to overshoot a local minimum so drastically that recovery is impossible.

**The Fix — `torch.nn.utils.clip_grad_norm_`:**

This function computes the **global $\ell_2$ norm** of all gradients in the model:

$$\|g\|_2 = \sqrt{\sum_{p \in \text{params}} \sum_{j} g_{p,j}^2}$$

If this norm exceeds a threshold $\theta$ (typically `1.0` or `5.0`), it **rescales all gradients** uniformly so that the norm equals $\theta$:

$$g \leftarrow g \cdot \frac{\theta}{\|g\|_2} \quad \text{if} \quad \|g\|_2 > \theta$$

**Critical API Understanding:**
- It must be called **after** `loss.backward()` (so gradients exist) and **before** `optimizer.step()` (so the clipped gradients are used).
- It clips **in-place** on the `.grad` attributes of parameters.
- It returns the norm **before** clipping — monitoring this value reveals if your model is numerically unstable.

```python
loss.backward()
# ↓ THE MAGIC HAPPENS HERE — always between backward() and step()
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()
```

🎓 *Socratic Probe: Gradient clipping preserves the **direction** of the gradient update and only rescales its **magnitude**. Does this mean the training trajectory is qualitatively identical to unclipped training? Think carefully about what "direction" means in a high-dimensional parameter space when the gradient norm is enormous.*

In [ ]:
import torch.optim as optim
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# 1. Synthetic Dataset
training_data = [
    ("The cat ate the mouse".split(), ["DET", "NN", "VB", "DET", "NN"]),
    ("A quick brown fox".split(), ["DET", "JJ", "JJ", "NN"]),
    ("Jumps over the lazy dog".split(), ["VB", "IN", "DET", "JJ", "NN"]),
    ("The clever professor taught the class".split(), ["DET", "JJ", "NN", "VB", "DET", "NN"])
]

word_to_ix = {"<PAD>": 0}
for sent, tags in training_data:
    for word in sent:
        if word not in word_to_ix: word_to_ix[word] = len(word_to_ix)

tag_to_ix = {"<PAD>": 0, "DET": 1, "NN": 2, "VB": 3, "JJ": 4, "IN": 5}

def prepare_sequence(seq, to_ix):
    idxs = [to_ix[w] for w in seq]
    return torch.tensor(idxs, dtype=torch.long)

print(f"Vocab size: {len(word_to_ix)}")

### 4.2 · Tensor Memory Layout: Packed Padded Sequences

In our POS tagger, we deal with sequences of different lengths (e.g., 5 tokens, 10 tokens, 100 tokens). To process them in a single batch, we use **Padding**—filling the gaps with zeros so every sequence matches the `max_len` of the batch.

#### 1. The "Padding Noise" Problem
While padding makes the tensors "square" and easy for the GPU to handle, it introduces two major engineering flaws:
- **Computational Waste (FLOPs)**: If your batch has one sequence of length 100 and nine sequences of length 10, the LSTM will spend 90% of its time calculating math on zeros.
- **Hidden State Corruption**: An LSTM is a memory-based model. If it processes 90 steps of pure zeros after the actual text, the final hidden state $h_T$ will be "washed out" or corrupted by the blank inputs. For tasks like sentence classification (where we use the final state), this decays performance significantly.

#### 2. The Solution: Packing Sequences
PyTorch provides `torch.nn.utils.rnn.pack_padded_sequence` to solve this. Instead of a "square" tensor, it transforms the batch into a **PackedSequence** object.

**Under the Hood:**
1. **Sorting**: It sorts the batch by length in descending order.
2. **Flattening**: It flattens the data by **timestep**, not by sequence.
3. **Batch-Size Tracking**: It tracks how many sequences are still "active" at each step. 
   - At step 1, batch size might be 32.
   - At step 50, batch size might drop to 12.
   - At step 100, batch size might be 1.

The C++ backend of the LSTM then only computes the matrix multiplications for the *active* items in the batch at each step. This eliminates the "padding noise" and ensures the hidden state stops updating exactly when the sequence ends.

#### 3. Engineering Pattern: Pack → Process → Unpack
```python
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# 1. Setup sample data: 3 sequences of different lengths
embeddings = torch.randn(3, 5, 10) # (Batch=3, Seq=5, Dim=10)
lengths = torch.tensor([5, 3, 2])    # Actual lengths

# 2. Pack the sequence
# Using enforce_sorted=False allows us to pass unsorted data
packed_input = pack_padded_sequence(embeddings, lengths, 
                                    batch_first=True, enforce_sorted=False)

# 3. Pass through RNN/LSTM
lstm = nn.LSTM(input_size=10, hidden_size=20, batch_first=True)
packed_output, (h, c) = lstm(packed_input)

# 4. Unpack (Transform back to a padded tensor)
output, _ = pad_packed_sequence(packed_output, batch_first=True)

print(f"Original shape: {embeddings.shape}")
print(f"Unpacked shape: {output.shape}")
```

> **🎓 Masterclass Insight:** Always use `enforce_sorted=False` in modern PyTorch (>= 1.1) to avoid manual sorting/unsorting of labels, which is a common source of bugs in sequence pipelines.


In [ ]:
class BiLSTMPOSTagger(nn.Module):
    """Production-grade BiLSTM for sequence tagging.
    
    Uses pack_padded_sequence to skip PAD token computation,
    yielding significant speedups on variable-length batches.
    """
    def __init__(self, vocab_size: int, tagset_size: int, 
                 embedding_dim: int, hidden_dim: int):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # The BiLSTM
        self.lstm = nn.LSTM(embedding_dim, hidden_dim,
                           num_layers=1, bidirectional=True, batch_first=True)
        
        # Dropout for regularization — standard in production LSTM pipelines
        self.dropout = nn.Dropout(p=0.3)
        
        # Classifier: hidden_dim * 2 because it's bidirectional
        self.hidden2tag = nn.Linear(hidden_dim * 2, tagset_size)

    def forward(self, sentence_padded: torch.Tensor, lengths: list) -> torch.Tensor:
        embeds = self.dropout(self.word_embeddings(sentence_padded))
        
        # PACKING: Skip computation on pads
        packed_embeds = pack_padded_sequence(embeds, lengths,
                                            batch_first=True, enforce_sorted=False)
        
        lstm_out, _ = self.lstm(packed_embeds)
        
        # UNPACKING
        output, _ = pad_packed_sequence(lstm_out, batch_first=True)
        output = self.dropout(output)
        
        tag_space = self.hidden2tag(output)
        return tag_space

In [ ]:
EMBEDDING_DIM = 32
HIDDEN_DIM   = 32
MAX_GRAD_NORM = 1.0  # The gradient clipping threshold — a sacred hyperparameter

model = BiLSTMPOSTagger(len(word_to_ix), len(tag_to_ix), EMBEDDING_DIM, HIDDEN_DIM)
loss_function = nn.CrossEntropyLoss(ignore_index=0)  # Ignore <PAD>
optimizer = optim.Adam(model.parameters(), lr=0.1)

# Prepare mini-batch (normally handled by DataLoader & Collator)
def get_batch():
    sents  = [prepare_sequence(s, word_to_ix) for s, t in training_data]
    tags   = [prepare_sequence(t, tag_to_ix)  for s, t in training_data]
    lengths = [len(s) for s in sents]
    
    # Padding
    padded_sents = torch.nn.utils.rnn.pad_sequence(sents, batch_first=True, padding_value=0)
    padded_tags  = torch.nn.utils.rnn.pad_sequence(tags,  batch_first=True, padding_value=0)
    
    return padded_sents, padded_tags, lengths

grad_norms = []  # For plotting training stability

print("Starting Training (with Gradient Clipping)...")
print("-" * 55)
print(f"{'Epoch':>5} | {'Loss':>8} | {'Grad Norm (pre-clip)':>20}")
print("-" * 55)

for epoch in range(50):
    model.train()
    model.zero_grad()
    sents, tags, lengths = get_batch()
    
    tag_scores = model(sents, lengths)
    
    # Reshape for CrossEntropyLoss: (Batch * Seq, Classes)
    loss = loss_function(tag_scores.view(-1, len(tag_to_ix)), tags.view(-1))
    
    # --- STEP 1: Compute gradients ---
    loss.backward()
    
    # --- STEP 2: CLIP before the optimizer corrupts the model ---
    # clip_grad_norm_ returns the TOTAL gradient norm BEFORE clipping.
    # Monitor this: if it's consistently >> MAX_GRAD_NORM, your LR is too high
    # or your model architecture is unstable.
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
    grad_norms.append(grad_norm.item())
    
    # --- STEP 3: Apply the clipped gradients ---
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f"{epoch+1:>5} | {loss.item():>8.4f} | {grad_norm.item():>20.4f}")

print("-" * 55)
print("\n--- TEST IT ---")
model.eval()
with torch.no_grad():
    test_sent, _, test_len = get_batch()
    scores = model(test_sent, test_len)
    predictions = torch.argmax(scores, dim=2)
    print(f"Example prediction for 'The cat ate the mouse':")
    predicted_tags = [list(tag_to_ix.keys())[idx] for idx in predictions[0] if idx != 0]
    print(predicted_tags)

In [ ]:
# Visualize the pre-clip gradient norm over training
# A healthy training run shows high norms early (exploration) that
# decay as the model converges. A flat line at MAX_GRAD_NORM signals
# the clip is actively firing — a useful diagnostic signal.

plt.figure(figsize=(10, 4))
plt.plot(grad_norms, color='#9b59b6', lw=2, label='Pre-clip Gradient Norm')
plt.axhline(y=MAX_GRAD_NORM, color='#e74c3c', linestyle='--', lw=1.5,
            label=f'Clip Threshold = {MAX_GRAD_NORM}')
plt.xlabel('Training Step')
plt.ylabel('Global Gradient L2 Norm')
plt.title('Gradient Norm Monitoring — Clip Fires When Norm > Threshold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

clip_events = sum(1 for n in grad_norms if n > MAX_GRAD_NORM)
print(f"Clip fired in {clip_events}/{len(grad_norms)} steps ({100*clip_events/len(grad_norms):.1f}% of training).")

### 4.4 · The Missing Link for Sequence Tagging: Conditional Random Fields (CRFs)

### The Structural Flaw in BiLSTM Taggers

In our BiLSTM POS tagger above, notice the final classification layer (`self.hidden2tag`). For a given sequence length $T$, it produces $T$ independent probability distributions over the tagset. We then used `torch.argmax` to pick the best tag at each step $t$, **entirely independent** of the tag chosen at step $t-1$.

For simple POS tagging, this is mostly fine because the BiLSTM's hidden state $h_t$ encodes strong contextual clues. However, for strict sequence tagging schemes like **NER with BIO tracking** (Begin-Inside-Outside), this independent prediction is **catastrophic**.

Consider the labels for Named Entity Recognition:
- `B-PER` (Begin Person)
- `I-PER` (Inside Person)
- `B-ORG` (Begin Organization)
- `I-ORG` (Inside Organization)
- `O` (Outside)

By definition, an `I-ORG` tag **cannot** legally follow a `B-PER` tag. It violates the grammar of the BIO scheme. But a pure BiLSTM doesn't strictly know this rule—it might assign high probability to `B-PER` at $t$ and high probability to `I-ORG` at $t+1$ because the independent softmax doesn't constrain the sequence structure. The predictions lack **structural consistency**.

### Enter the CRF Layer

A Conditional Random Field (CRF) solves this by modeling the **joint probability** of the entire sequence of tags, rather than predicting them independently.

Instead of outputting raw softmax probabilities, the BiLSTM passes its outputs (called **Emission Scores**) to the CRF layer. The CRF layer maintains a **Transition Matrix** of shape `(num_tags, num_tags)` which it learns during training.

The total score of a sequence of tags $y = (y_1, y_2, \dots, y_T)$ for a sentence $x$ is defined as:

$$\text{Score}(x, y) = \sum_{i=1}^T \text{Emission}(x_i, y_i) + \sum_{i=1}^{T} \text{Transition}(y_{i-1}, y_i)$$

During training, the CRF learns that `Transition(B-PER, I-ORG)` should have a massive negative penalty ($-\infty$). This elegantly guarantees that the network will never produce an invalid sequence of tags.

### Inference: Viterbi Decoding

During inference, we cannot just confidently pick the best tag at each step. Because of the transition scores, picking the 2nd-best emission at step $t$ might unlock an extremely high transition score at step $t+1$. We must find the path with the **global maximum score**.

Exploring all possible tag sequences takes $O(|V|^T)$, which is intractable. Instead, we use the **Viterbi Algorithm**, an elegant dynamic programming technique that finds the optimal path in $O(T \cdot |V|^2)$.

In [ ]:
def viterbi_decode(emissions: torch.Tensor, transitions: torch.Tensor):
    """
    Pedagogical Viterbi Decoding (Pseudo-code style implementation).
    
    Args:
        emissions:   (seq_len, num_tags) - The outputs from the BiLSTM (often called logits)
        transitions: (num_tags, num_tags) - The learnable CRF transition matrix.
                     transitions[i, j] is the score of transitioning from tag i to tag j.
    
    Returns:
        best_path: List of the optimal tag indices for the sequence.
    """
    seq_len, num_tags = emissions.shape
    
    # viterbi_vars[j] stores the maximum score of a path ending in tag j at the current step.
    # Initialize with the emission scores of the first word.
    viterbi_vars = emissions[0]
    
    # We need to keep track of the "backpointers" to reconstruct the best path at the end.
    backpointers = []
    
    # Forward pass: iterate through the sequence starting from the second word
    for i in range(1, seq_len):
        bptrs_t = []          # Backpointers for this step
        viterbi_vars_t = []   # New viterbi_vars for this step
        
        for next_tag in range(num_tags):
            # Score of transitioning from ANY previous tag to the current `next_tag`:
            # score = previous path score + transition score + emission score
            next_tag_var = viterbi_vars + transitions[:, next_tag] + emissions[i, next_tag]
            
            # The Viterbi insight: we only care about the MAXIMUM path leading to `next_tag`.
            best_prev_tag = torch.argmax(next_tag_var)
            bptrs_t.append(best_prev_tag.item())
            viterbi_vars_t.append(next_tag_var[best_prev_tag].item())
            
        viterbi_vars = torch.tensor(viterbi_vars_t)
        backpointers.append(bptrs_t)
        
    # End of sequence. Find the absolute best final tag.
    best_final_tag = torch.argmax(viterbi_vars).item()
    best_path = [best_final_tag]
    
    # Backtrack through the backpointers to reconstruct the optimal sequence
    for bptrs_t in reversed(backpointers):
        best_final_tag = bptrs_t[best_final_tag]
        best_path.append(best_final_tag)
        
    # Reverse to get the path from start to finish
    best_path.reverse()
    return best_path

print("1. The BiLSTM produces context-aware token features.")
print("2. The CRF Transition log-probabilities enforce grammar-like constraints (e.g., I-ORG must follow B-ORG).")
print("3. Viterbi decoding guarantees we find the globally optimal tag sequence through the trellis.")

---
## 5 · Unaligned Sequences: Connectionist Temporal Classification (CTC) Loss

### 🎓 Socratic Deep Check
> *"We've seen how Seq2Seq models use bottlenecks (Encoder-Decoder) to handle different lengths, and BiLSTMs use 1-to-1 tagging for tasks like POS. But what about Speech-to-Text or Handwriting OCR? Your audio input might be 2,000 frames long, while the transcript is only 50 characters. We know the sequence is monotonic (the characters appear in order), but we have **no idea** which frame corresponds to which character. How can we train a model to maximize likelihood when the alignment is 'fuzzy' and the lengths are vastly different?"*

### The Alignment Problem
In many real-world sequence tasks, we lack a ground-truth alignment between input and output. Labeling exactly which millisecond of audio corresponds to the start of the 'k' sound in 'cat' is extremely difficult and noisy. We need a way to train the network to find the alignment itself.

### Connectionist Temporal Classification (CTC)
Introduced by Graves et al. in 2006, **CTC Loss** allows an RNN to produce a sequence that is longer than the target text and collapse it into the correct string. 

#### 1. The Invention of the "Blank Token" (`-`)
CTC introduces a special **Blank Token** (not to be confused with a space). If the model output is `[c, c, -, a, -, -, t]`, we apply two rules to collapse it:
1.  **Remove Adjacent Repetitions**: `[c, c, a, t]` $\rightarrow$ `[c, a, t]`
2.  **Strip Blanks**: `[c, a, t]` $\rightarrow$ `"cat"` 

Wait—why do we need the blank? If the word was "hello" (double 'l'), without the blank, `[h, e, l, l, o]` would collapse into `"helo"`. The blank token acts as a mandatory separator for repeated characters.

#### 2. The Mechanism: Marginalizing over Paths
For a given input sequence $X$, there are millions of ways the model could output a sequence that collapses to `"cat"` (e.g., `cccaaat`, `c-a-t`, `---c-a-t---`). 

The **CTC Loss Function** uses **Dynamic Programming** (the forward-backward algorithm) to efficiently sum the probabilities of **every possible valid alignment path**. We then maximize the log-likelihood of this entire sum:

$$L = -\log \sum_{\pi \in \mathcal{B}^{-1}(y)} P(\pi | X)$$

Where $\mathcal{B}$ is the collapse operator. By maximizing this sum, the model implicitly learns the most efficient way to align the audio frames or image slices to the target characters without ever being shown the "correct" timing.


## 6 · 1D CNNs for Text (TextCNN): When Parallelism Beats Memory

We have established that RNNs are powerful sequence models — but they are inherently **serial**. The hidden state $h_t$ depends on $h_{t-1}$, creating an inviolable step-by-step dependency chain. On a modern GPU with thousands of CUDA cores, this is a cardinal sin: the hardware sits mostly idle while the model processes one token at a time.

🎓 *Socratic Question: Before reading on — can you think of an architecture that already processes spatial data (like images) in a fully parallel manner? How might you adapt it to process the "time" dimension of text?*

### The Key Insight: Text as a 1D Signal

Consider a sentence represented as a matrix $X \in \mathbb{R}^{T \times d}$ (T tokens, d-dimensional embeddings). This is structurally **identical** to a 1D time-series signal of length $T$ with $d$ channels.

A **1D Convolution** sweeps a filter $W \in \mathbb{R}^{k \times d}$ of width $k$ (the kernel size) across all $T$ positions simultaneously:

$$c_i = \phi\left(\sum_{j=0}^{k-1} W_j \cdot x_{i+j}^T + b\right), \quad i = 1 \ldots T-k+1$$

This is an **n-gram feature detector**:
- A kernel of size **2** detects **bigram** patterns (e.g., "not happy", "very good").
- A kernel of size **3** detects **trigram** patterns.
- A kernel of size **5** detects **5-gram** patterns.

After convolution, **Global Max-Pooling** collapses the sequence dimension entirely, extracting the *most activated* feature for each filter across the **entire sentence**:

$$\hat{c} = \max_{i} c_i$$

### TextCNN Architecture (Kim, 2014)

The seminal TextCNN architecture ("Convolutional Neural Networks for Sentence Classification") uses **multiple filter sizes in parallel**, then concatenates their pooled outputs before a classifier:

```
Sentence Embeddings (T × d)
          |
   ┌──────┼──────┐
   ↓      ↓      ↓
 Conv1D  Conv1D Conv1D    ← Kernel sizes: 2, 3, 5
 (k=2)  (k=3)  (k=5)     ← Multiple filters each
   ↓      ↓      ↓
 MaxPool MaxPool MaxPool  ← Global Max-Pooling
   ↓      ↓      ↓
   └──────┼──────┘
          ↓
      Concatenate
          ↓
        Dense + Softmax
```

### ⚡ Speed & Parallelism: TextCNN vs LSTM

| Property | LSTM / BiLSTM | TextCNN |
|----------|--------------|--------|
| **Computation** | Sequential (step-by-step) | Fully Parallel |
| **Long-Range Dependencies** | Excellent (by design) | Poor (bounded by kernel size) |
| **Training Speed** | Slow | Very Fast |
| **Inference Latency** | Higher | Very Low |
| **Local Feature Detection** | Implicit | Explicit (by design) |
| **Task Suitability** | Sequence tagging, generation | Sentence classification |
| **GPU Utilization** | Low (~serial bottleneck) | High (~full parallelism) |

**The takeaway:** TextCNN is not "better" or "worse" than LSTM — it solves a *different* structural problem. For **sentence-level classification** (sentiment, topic, intent), TextCNN often matches or beats BiLSTMs at a fraction of the computational cost. For tasks requiring **fine-grained state over long distances** (NER, POS tagging, machine translation), LSTMs win.

In [ ]:
class TextCNN(nn.Module):
    """Elite Implementation: TextCNN for Text Classification.
    
    Implements the multi-kernel-size architecture from Kim (2014) —
    'Convolutional Neural Networks for Sentence Classification' — using
    parallel 1D convolution branches with global max-pooling.
    
    The key trick is using Conv1d with in_channels=embed_dim and treating
    each token's embedding as the 'spatial' input. The kernel slides
    across the sequence (time) dimension.
    
    Args:
        vocab_size:  Size of the vocabulary.
        embed_dim:   Embedding dimensionality.
        num_classes: Number of output classes.
        num_filters: Number of convolutional filters per kernel size.
        kernel_sizes: List of kernel widths (n-gram sizes to detect).
        dropout:     Dropout probability for regularization.
    """
    def __init__(
        self,
        vocab_size:   int,
        embed_dim:    int,
        num_classes:  int,
        num_filters:  int       = 128,
        kernel_sizes: list      = [2, 3, 5],
        dropout:      float     = 0.5,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # One Conv1d branch per kernel size.
        # Conv1d expects input shape: (Batch, Channels, Length)
        # Here, Channels = embed_dim, Length = seq_len
        # After convolution: (Batch, num_filters, seq_len - k + 1)
        self.conv_branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(in_channels=embed_dim, out_channels=num_filters,
                          kernel_size=k, padding=0),
                nn.ReLU(),
                # Global Max-Pooling: collapse the entire sequence dimension
                nn.AdaptiveMaxPool1d(1),  # → (Batch, num_filters, 1)
            )
            for k in kernel_sizes
        ])
        
        self.dropout    = nn.Dropout(dropout)
        # Final classifier: num_filters * len(kernel_sizes) features
        self.classifier = nn.Linear(num_filters * len(kernel_sizes), num_classes)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Token index tensor of shape (Batch, Seq_Len)
        Returns:
            Logits of shape (Batch, num_classes)
        """
        # (Batch, Seq_Len) → (Batch, Seq_Len, embed_dim)
        embedded = self.embedding(x)
        
        # Conv1d expects (Batch, Channels, Length), not (Batch, Length, Channels)
        # So we permute: (Batch, Seq_Len, embed_dim) → (Batch, embed_dim, Seq_Len)
        embedded = embedded.permute(0, 2, 1)
        
        # Apply each conv branch in parallel, squeeze the pooled dimension
        # Each branch: (Batch, num_filters, 1) → (Batch, num_filters)
        branch_outputs = [branch(embedded).squeeze(2) for branch in self.conv_branches]
        
        # Concatenate all branch features
        # (Batch, num_filters * len(kernel_sizes))
        pooled = torch.cat(branch_outputs, dim=1)
        
        out = self.classifier(self.dropout(pooled))
        return out


# ──────────────────────────────────────────────
# Demonstration: Sentiment Classification
# ──────────────────────────────────────────────
VOCAB_SIZE_DEMO  = 1000
EMBED_DIM_DEMO   = 64
NUM_CLASSES_DEMO = 2    # Positive / Negative
SEQ_LEN_DEMO     = 30   # Tokens per sentence
BATCH_SIZE_DEMO  = 16

cnn_model = TextCNN(
    vocab_size=VOCAB_SIZE_DEMO,
    embed_dim=EMBED_DIM_DEMO,
    num_classes=NUM_CLASSES_DEMO,
    num_filters=128,
    kernel_sizes=[2, 3, 5],
)

# Dummy batch
dummy_input = torch.randint(1, VOCAB_SIZE_DEMO, (BATCH_SIZE_DEMO, SEQ_LEN_DEMO))
logits = cnn_model(dummy_input)

print(f"TextCNN Architecture:")
print(f"  Input shape:  {tuple(dummy_input.shape)}     (Batch, Seq_Len)")
print(f"  Output shape: {tuple(logits.shape)}       (Batch, Classes)")

total_cnn_params = sum(p.numel() for p in cnn_model.parameters())
print(f"  Total parameters: {total_cnn_params:,}")
print(f"  Kernel branches: bigram (k=2), trigram (k=3), 5-gram (k=5)")

In [ ]:
# ──────────────────────────────────────────────────────
# Micro-Benchmark: TextCNN vs BiLSTM Forward-Pass Speed
# ──────────────────────────────────────────────────────
import time

VOCAB_SIZE_BM = 1000
EMBED_DIM_BM  = 64
HIDDEN_DIM_BM = 64
SEQ_LEN_BM    = 50
BATCH_SIZE_BM = 32
N_RUNS        = 100

# TextCNN
bm_cnn = TextCNN(VOCAB_SIZE_BM, EMBED_DIM_BM, num_classes=2,
                 num_filters=64, kernel_sizes=[2, 3, 5]).eval()

# BiLSTM (for classification — take last hidden state)
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.fc   = nn.Linear(hidden_dim * 2, num_classes)
    def forward(self, x):
        out, _ = self.lstm(self.emb(x))
        return self.fc(out[:, -1, :])  # Last token's hidden state

bm_lstm = BiLSTMClassifier(VOCAB_SIZE_BM, EMBED_DIM_BM, HIDDEN_DIM_BM, 2).eval()

dummy_bm = torch.randint(1, VOCAB_SIZE_BM, (BATCH_SIZE_BM, SEQ_LEN_BM))

# Warm-up
for _ in range(5):
    with torch.no_grad():
        _ = bm_cnn(dummy_bm)
        _ = bm_lstm(dummy_bm)

# Time TextCNN
t0 = time.perf_counter()
for _ in range(N_RUNS):
    with torch.no_grad():
        _ = bm_cnn(dummy_bm)
cnn_time = (time.perf_counter() - t0) / N_RUNS * 1000

# Time BiLSTM
t0 = time.perf_counter()
for _ in range(N_RUNS):
    with torch.no_grad():
        _ = bm_lstm(dummy_bm)
lstm_time = (time.perf_counter() - t0) / N_RUNS * 1000

print("\n{'='*45}")
print(f"  Micro-Benchmark: Forward Pass (CPU)")
print(f"  Batch={BATCH_SIZE_BM}, SeqLen={SEQ_LEN_BM}, EmbDim={EMBED_DIM_BM}")
print(f"  {'='*40}")
print(f"  TextCNN  : {cnn_time:6.2f} ms/batch")
print(f"  BiLSTM   : {lstm_time:6.2f} ms/batch")
print(f"  Speedup  : {lstm_time/cnn_time:.2f}x faster (TextCNN)")
print(f"{'='*45}")
print("\n  [Note: GPU speedup is typically even more dramatic")
print("   because CNN ops saturate GPU cores while LSTM remains serial.]")

### 🔬 TextCNN: Worked Example — What Does Each Kernel Detect?

Consider the sentence: `"The film was not at all good"`

With a **bigram filter** ($k=2$), the convolution slides over pairs:
- `["The", "film"]`, `["film", "was"]`, `["was", "not"]`, **`["not", "at"]`**, **`["at", "all"]`**, **`["all", "good"]`**

A well-trained filter might fire strongly on `"not good"` or `"not at all good"` — learning a **negation n-gram** that's a strong marker of negative sentiment.

With Global Max-Pooling, only the **strongest activation** across all positions survives — so the model captures **whether** a pattern exists anywhere in the sentence, regardless of position.

🎓 *Socratic Question: TextCNN's max-pooling is position-invariant — it doesn't care where in the sentence the pattern appears. Is this always an advantage? Can you think of a task where the Position of a specific n-gram is critical to the correct answer?*

*(Hint: Consider 'the start of the sentence' vs 'the end of the sentence' in discourse structure or syntactic parsing.)*

## 7 · Performance & The Evolution to Attention

### The Serial Bottleneck
While LSTMs and GRUs fixed the **vanishing gradient**, they did not fix the **computational efficiency** problem.

Modern GPUs have thousands of cores.
- **CNNs** are fully parallelable across space.
- **Transformers** are fully parallelable across time.
- **RNNs** are **strictly sequential**. You cannot compute $h_t$ until $h_{t-1}$ is finished. This means TFLOPS of GPU power sit idle as the model processes one word at a time.

### The Architecture Spectrum

```
        ← MORE MEMORY                    MORE SPEED →
        ← LONG RANGE                    LOCAL RANGE →

  Vanilla RNN ──── LSTM/GRU ──── Transformer ──── TextCNN
       ↑               ↑               ↑              ↑
   Pathological   Industry Horse   SOTA NLP    Fast Classifier
   (≤10 steps)   (100-500 steps)  (unbounded)  (sentence-level)
```

No single architecture dominates all tasks. The expert practitioner chooses based on:
1. **Sequence length**: Long sequences → Transformer or LSTM. Short sentences → TextCNN.
2. **Latency requirements**: Real-time inference → TextCNN or quantized Transformer.
3. **Data size**: Small dataset → LSTM (fewer params). Large dataset → Transformer.
4. **Task structure**: Classification → TextCNN competes strongly. Token-level → LSTM or Transformer.

---

## ✅ Knowledge Check

### Q1: The Forget Gate
If you initialize an LSTM's forget gate biases to a high positive value (e.g., +1.0 or +2.0) during training, what problem are you trying to mitigate?

<details><summary>👉 Answer</summary>
You are mitigating the vanishing gradient. By setting $b_f$ high, $f_t \approx 1$ initially, which forces the model to keep most of the memory in the early stages of training, allowing gradients to flow further back. This initialization trick is well-established in the LSTM literature and is sometimes called "chrono initialization".
</details>

### Q2: Bidirectionality
Can you use a BiLSTM for a real-time speech-to-text system where latency must be < 100ms?

<details><summary>👉 Answer</summary>
Technically no (or with great difficulty). A BiLSTM requires the **entire** sequence to be finished before it can compute the backward hidden states. In real-time systems, you haven't heard the end of the sentence yet. You would need to use lookahead-windows or strictly unidirectional models.
</details>

### Q3: Gradient Clipping
You are training an LSTM and notice that `clip_grad_norm_` is firing every single step (the pre-clip norm is always 5–10x the threshold). What does this indicate and what are two remedies?

<details><summary>👉 Answer</summary>
This indicates that the model is consistently producing very large gradients — the training is **numerically unstable**. Two remedies:

1. **Reduce the learning rate**: The optimizer is taking steps that are too large, causing oscillating large gradients.
2. **Increase the clip threshold**: If the norms are just slightly above the threshold and the model is learning well, a more permissive threshold (e.g., 5.0 instead of 1.0) can help.

A more architectural remedy is adding **layer normalization** or using **gradient accumulation** to smooth the per-step gradient estimates.
</details>

### Q4: TextCNN vs LSTM
A colleague proposes replacing a BiLSTM NER (Named Entity Recognition) tagger with a TextCNN for 5x speed improvement. What is the fundamental architectural objection?

<details><summary>👉 Answer</summary>
NER requires **token-level** predictions (a label for **every** word). TextCNN with global max-pooling collapses the sequence dimension entirely into a single vector — it cannot produce per-token outputs. You would need to remove the global pooling and use local (stride-1) pooling instead, but then you lose the model's ability to integrate features from the entire sentence. The BiLSTM's hidden state at each position encodes the full left and right context — critical for resolving entity boundary ambiguities. TextCNN is best suited for **sentence-level** classification, not token-level tagging.
</details>

### Q5: GRU vs LSTM
Which architecture is generally more prone to exploding gradients (without clipping)?

<details><summary>👉 Answer</summary>
Both are safer than Vanilla RNNs, but they can both produce large activations if not properly normalized. However, the exploding gradient is usually solved by **Gradient Clipping**, which is non-negotiable for all recurrent network training. In practice, the LSTM's independent cell state provides a slightly more stable gradient path than the GRU's single recurrent state.
</details>

---

**Next →** `16_04_sequence_to_sequence_and_attention.ipynb` — NLP Masterclass 04: Sequence-to-Sequence, Attention, Decoding & Evaluation